In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from modelscope import AutoModelForCausalLM, AutoTokenizer
import torch
import re

# The model can be downloaded on https://huggingface.co/Qwen/Qwen3-8B
chat_model_path = ".\Qwenchat8B"
tokenizer = AutoTokenizer.from_pretrained(chat_model_path)
chat_model = AutoModelForCausalLM.from_pretrained(
    chat_model_path,
    torch_dtype="auto",
    device_map="auto"
)

# The model can be downloaded on https://huggingface.co/Qwen/Qwen3-Embedding-8B
embed_model = SentenceTransformer(r".\Qwen8B_embedding")
csv_path = "19年天气加事件加搜索指数0911.csv"

# 读取每日数据（日期、星期、天气、降雨量、事件、前一天搜索指数、截至前一天历史平均搜索指数）
try:
    df = pd.read_csv(csv_path, encoding="utf-8")
except UnicodeDecodeError:
    try:
        df = pd.read_csv(csv_path, encoding="gbk")
    except UnicodeDecodeError:
        df = pd.read_csv(csv_path, encoding="gb18030")

assert df.shape[1] >= 7, "CSV至少需要七列（日期、星期、天气、降雨量、事件、前一天搜索指数、截至前一天历史平均搜索指数）"

# 编码结果保存
embedding_results = []

print("正在进行每天事件的语言理解与编码...")

# 遍历每一天
for i, row in tqdm(df.iterrows(), total=len(df)):
    date = str(row.iloc[0])
    weekday = str(row.iloc[1])
    weather = str(row.iloc[2])
    rainfall = str(row.iloc[3])
    event = str(row.iloc[4]).strip() if pd.notna(row.iloc[4]) else ""
    search_index = str(row.iloc[5]) if pd.notna(row.iloc[5]) else "无"
    search_index_mean_until_yesterday = (
        str(row.iloc[6]) if pd.notna(row.iloc[6]) else "无"
    )
    location = "北京市"

    if not event:
        user_prompt = f"""
已知以下日常出行信息：

- 时间：{date}
- 地点：{location}
- 天气：{weather}
- 降雨量：{rainfall}
- 星期：{weekday}
- 事件：无重大事件
- 前一天景区平均搜索指数：{search_index}
- 截至前一天的历史平均搜索指数：{search_index_mean_until_yesterday}

请分析当天在没有重大事件情况下，天气、降雨量、搜索热度和星期可能对城市中景点人流量的影响：
1. 请结合天气、降雨量、星期和搜索热度，说明当天人流量可能的变化趋势。
2. 总体来看，对人流量的影响是正向增加还是负向减少，程度如何？

请以简洁自然语言进行描述。
"""
    else:
        user_prompt = f"""
已知以下事件信息：

- 时间：{date}
- 地点：{location}
- 事件：{event}
- 天气：{weather}
- 降雨量：{rainfall}
- 星期：{weekday}
- 前一天景区平均搜索指数：{search_index}
- 截至前一天的历史平均搜索指数：{search_index_mean_until_yesterday}

请分析该事件对城市中景点人流量的影响：
1. 请结合事件类型、天气、降雨量、星期和搜索热度，说明该事件可能对人流量产生的叠加影响。
2. 总体来看，该事件对人流量的影响是正向增加还是负向减少，程度如何？

请以简洁自然语言进行描述。
"""

    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True 
    )

    inputs = tokenizer([text], return_tensors="pt").to(chat_model.device)


    with torch.no_grad():
        output_ids = chat_model.generate(
            **inputs,
            max_new_tokens=2048,     
            do_sample=True,         
            temperature=0.5,         
            top_p=0.9,               
            repetition_penalty=1.1   
        )[0]
    
    generated_text = tokenizer.decode(
        output_ids[len(inputs.input_ids[0]):],
        skip_special_tokens=False  
    )

    content = re.sub(r"<think>.*?</think>", "", generated_text, flags=re.S)
    
    content = content.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()

    print(
        f"日期：{date}，事件：{event if event else '无'}\n"
        f"天气：{weather}，降雨量：{rainfall}，星期：{weekday}\n"
        f"前一天搜索指数：{search_index}，截至前一天历史均值：{search_index_mean_until_yesterday}\n"
        f"Chat模型回复：{content}\n"
        f"{'-' * 50}"
    )

    # 编码
    emb = embed_model.encode(content, convert_to_numpy=True)
    embedding_results.append(emb)

# 保存为 numpy 文件
embedding_array = np.stack(embedding_results, axis=0)
np.save("事件Chat理解嵌入20260530_含历史搜索均值.npy", embedding_array)

print(
    f"\n所有事件已分析并编码完成，保存为 "
    f"'事件Chat理解嵌入20260530_含历史搜索均值.npy'，维度为：{embedding_array.shape}"
)


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

正在进行每天事件的语言理解与编码...


  0%|                                                                                          | 0/365 [00:00<?, ?it/s]

日期：2019/1/1，事件：元旦节
天气：晴~多云，降雨量：0，星期：周二
前一天搜索指数：520.16，截至前一天历史均值：520.16
Chat模型回复：1. **叠加影响分析**  
   元旦节作为重要节日，通常会引发市民出游意愿提升，尤其在晴朗天气（晴~多云）和无降雨的情况下，户外活动更具吸引力。尽管为工作日（周二），但元旦属法定假日，多数人可灵活安排时间，抵消 weekday 的常规限流效应。同时，景区搜索指数与历史均值持平，表明公众兴趣未明显波动，但节日本身的传播力可能进一步刺激短期需求，形成“节日+好天气”的双重驱动。

2. **整体影响结论**  
   该事件对人流量呈现**正向增加**，程度较显著。元旦的节日属性、适宜天气及假日弹性，共同推动景点接待量高于平日，但需注意因非周末可能导致客流高峰集中于上午或下午时段，需合理分流管理。
--------------------------------------------------


...

日期：2019/12/30，事件：第五届大众冰雪北京公开赛
天气：晴，降雨量：0，星期：周一
前一天搜索指数：496.61，截至前一天历史均值：737.97
Chat模型回复：1. **叠加影响分析**  
   第五届大众冰雪北京公开赛作为冬季特色体育赛事，具有较强的吸引力，尤其在晴朗无雨的天气条件下，更利于户外冰雪活动开展。结合天气条件（晴/降雨量0）和活动性质（冰雪运动），该事件可能直接带动滑雪场、冰场等冰雪主题景点的人流增长。此外，尽管为工作日（周一），但赛事的娱乐性和季节性特征可能促使部分市民调整出行计划，形成短期集中客流。值得注意的是，前一日景区搜索指数（496.61）低于历史平均水平（737.97），表明前期关注度不足，赛事可能通过宣传和热点效应引发搜索热度回升，进一步放大人流增量。

2. **总体影响结论**  
   综合事件类型、天气条件及搜索热度变化，该事件对城市景点人流量的影响总体呈**正向增加趋势**。冰雪赛事的季节性需求、良好的天气条件以及活动本身的娱乐属性，将显著提升相关区域的客流量。但由于为工作日且前期搜索热度偏低，增幅可能较平日周末有所弱化，但仍可预期人流量较常规水平提高约**20%-30%**，具体需结合当日实时数据验证。
--------------------------------------------------


100%|█████████████████████████████████████████████████████████████████████████████▊| 364/365 [8:26:09<01:16, 76.04s/it]

日期：2019/12/31，事件：第五届大众冰雪北京公开赛
天气：晴，降雨量：0，星期：周二
前一天搜索指数：494.91，截至前一天历史均值：737.31
Chat模型回复：1. **叠加影响分析**  
   第五届大众冰雪北京公开赛作为冬季特色体育赛事，与晴朗无雨的天气形成积极叠加效应，利于户外活动开展。尽管为工作日（周二），但冰雪主题赛事本身具有较强吸引力，可能吸引本地居民及周边游客参与。同时，前一日景区搜索指数（494.91）虽低于历史平均水平（737.31），但赛事作为近期热点事件，可能通过宣传进一步刺激当日搜索热度，间接引导人流。综上，事件与天气条件共同构成正向驱动因素，抵消部分工作日的潜在负面影响。

2. **总体影响判断**  
   该事件对人流量呈现**正向增加趋势**。冰雪赛事的娱乐性和季节性特征、良好天气条件以及赛事本身的传播效应，预计会显著提升当日景点人流量。尽管工作日可能导致部分群体出行受限，但赛事吸引力和天气优势足以覆盖这一限制，最终人流量增幅应高于常规工作日水平，但具体程度需结合实际参与人数和景区承载能力进一步验证。
--------------------------------------------------


100%|██████████████████████████████████████████████████████████████████████████████| 365/365 [8:27:14<00:00, 83.38s/it]


所有事件已分析并编码完成，保存为 '事件Chat理解嵌入20260530_含历史搜索均值.npy'，维度为：(365, 4096)


In [3]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from modelscope import AutoModelForCausalLM, AutoTokenizer
import torch
import re

chat_model_path = "D:\\shenxi\\Qwen\\Qwenchat8B"
tokenizer = AutoTokenizer.from_pretrained(chat_model_path)
chat_model = AutoModelForCausalLM.from_pretrained(
    chat_model_path,
    torch_dtype="auto",
    device_map="auto"
)

embed_model = SentenceTransformer(r".\Qwen8B_embedding")
csv_path = "上海天气加事件+百度指数.csv"

try:
    df = pd.read_csv(csv_path, encoding="utf-8")
except UnicodeDecodeError:
    try:
        df = pd.read_csv(csv_path, encoding="gbk")
    except UnicodeDecodeError:
        df = pd.read_csv(csv_path, encoding="gb18030")

assert df.shape[1] >= 7, "CSV至少需要七列（日期、星期、天气、降雨量、事件、前一天搜索指数、截至前一天历史平均搜索指数）"

embedding_results = []

print("正在进行每天事件的语言理解与编码...")

# 遍历每一天
for i, row in tqdm(df.iterrows(), total=len(df)):
    date = str(row.iloc[0])
    weekday = str(row.iloc[1])
    weather = str(row.iloc[2])
    rainfall = str(row.iloc[3])
    event = str(row.iloc[4]).strip() if pd.notna(row.iloc[4]) else ""
    search_index = str(row.iloc[5]) if pd.notna(row.iloc[5]) else "无"
    search_index_mean_until_yesterday = (
        str(row.iloc[6]) if pd.notna(row.iloc[6]) else "无"
    )
    location = "上海市"

    if not event:
        user_prompt = f"""
已知以下日常出行信息：

- 时间：{date}
- 地点：{location}
- 天气：{weather}
- 降雨量：{rainfall}
- 星期：{weekday}
- 事件：无重大事件
- 前一天景区平均搜索指数：{search_index}
- 截至前一天的历史平均搜索指数：{search_index_mean_until_yesterday}

请分析当天在没有重大事件情况下，天气、降雨量、搜索热度和星期可能对城市中景点人流量的影响：
1. 请结合天气、降雨量、星期和搜索热度，说明当天人流量可能的变化趋势。
2. 总体来看，对人流量的影响是正向增加还是负向减少，程度如何？

请以简洁自然语言进行描述。
"""
    else:
        user_prompt = f"""
已知以下事件信息：

- 时间：{date}
- 地点：{location}
- 事件：{event}
- 天气：{weather}
- 降雨量：{rainfall}
- 星期：{weekday}
- 前一天景区平均搜索指数：{search_index}
- 截至前一天的历史平均搜索指数：{search_index_mean_until_yesterday}

请分析该事件对城市中景点人流量的影响：
1. 请结合事件类型、天气、降雨量、星期和搜索热度，说明该事件可能对人流量产生的叠加影响。
2. 总体来看，该事件对人流量的影响是正向增加还是负向减少，程度如何？

请以简洁自然语言进行描述。
"""

    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True 
    )

    inputs = tokenizer([text], return_tensors="pt").to(chat_model.device)


    with torch.no_grad():
        output_ids = chat_model.generate(
            **inputs,
            max_new_tokens=2048,     
            do_sample=True,         
            temperature=0.5,         
            top_p=0.9,               
            repetition_penalty=1.1   
        )[0]
    
    generated_text = tokenizer.decode(
        output_ids[len(inputs.input_ids[0]):],
        skip_special_tokens=False  # 【修改点3】：设为 False，保留 <think> 标签供下面正则使用
    )

    content = re.sub(r"<think>.*?</think>", "", generated_text, flags=re.S)
    
    content = content.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()

    print(
        f"日期：{date}，事件：{event if event else '无'}\n"
        f"天气：{weather}，降雨量：{rainfall}，星期：{weekday}\n"
        f"前一天搜索指数：{search_index}，截至前一天历史均值：{search_index_mean_until_yesterday}\n"
        f"Chat模型回复：{content}\n"
        f"{'-' * 50}"
    )

    # 编码
    emb = embed_model.encode(content, convert_to_numpy=True)
    embedding_results.append(emb)

# 保存为 numpy 文件
embedding_array = np.stack(embedding_results, axis=0)
np.save("事件Chat理解嵌入20260531_含历史搜索均值_SH.npy", embedding_array)

print(
    f"\n所有事件已分析并编码完成，保存为 "
    f"'事件Chat理解嵌入20260531_含历史搜索均值_SH.npy'，维度为：{embedding_array.shape}"
)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

正在进行每天事件的语言理解与编码...


  0%|                                                                                           | 0/92 [00:00<?, ?it/s]

日期：2018/7/1，事件：中国共产党成立纪念日（建党节）
天气：小雨~阴，降雨量：70，星期：周日
前一天搜索指数：675.15，截至前一天历史均值：675.15
Chat模型回复：1. **叠加影响分析**  
   - **事件类型**：建党节作为重大政治纪念日，通常会引发公众对红色文化、历史教育场所的关注，可能推动相关景点（如中共一大会址、革命纪念馆）的人流增加。  
   - **天气与降雨量**：小雨及70mm的降雨量可能抑制户外活动意愿，导致部分游客取消或推迟行程，尤其对依赖自然景观的景点形成负面影响。  
   - **星期因素**：周日为非工作日，居民闲暇时间增多，可能促进短途游或本地游览，但若天气不佳，这一效应会被弱化。  
   - **搜索热度**：前后两日搜索指数持平，表明公众对建党节的旅游关联性认知不足，或受天气影响导致搜索意愿未显著提升。  

2. **总体影响判断**  
   事件对人流量的影响呈现**小幅正向增长**，但程度有限。建党节的象征意义可能吸引部分人群前往红色景点，但降雨天气和稳定的搜索热度限制了整体增幅。若无特殊宣传或主题活动，实际人流量可能仅略高于平日，且主要集中在室内场馆，户外景点或面临轻微下滑。
--------------------------------------------------


...

日期：2018/9/29，事件：无
天气：多云~阴，降雨量：0，星期：周六
前一天搜索指数：782.23，截至前一天历史均值：802.07
Chat模型回复：1. **人流量变化趋势**  
   当天天气为多云转阴且无降雨，属于适宜户外活动的条件，可能适度提升市民出游意愿。作为周末（周六），休闲需求通常较高，叠加无重大事件干扰，理论上有助于人流量维持或小幅增长。然而，前一日景区搜索指数（782.23）略低于历史同期均值（802.07），表明前期热度较平稳，可能对当日人流形成一定抑制作用。综合来看，天气与周末的正向刺激作用或略强于搜索指数的微弱下滑，预计人流量将呈现**温和上升趋势**。

2. **总体影响方向与程度**  
   整体而言，天气、周末属性对人流量的正向拉动作用占主导地位，但因搜索指数未达历史峰值，增幅可能有限。因此，**人流量预计将小幅增加**（如增长5%-10%），属于中等偏弱的正向变动，主要受制于前期搜索热度的边际回落。
--------------------------------------------------


 99%|███████████████████████████████████████████████████████████████████████████████▏| 91/92 [1:50:52<01:06, 66.46s/it]

日期：2018/9/30，事件：无
天气：多云~晴，降雨量：0，星期：周日
前一天搜索指数：904.68，截至前一天历史均值：803.19
Chat模型回复：1. **人流量变化趋势分析**  
   - **天气与降雨量**：多云转晴的天气条件适宜户外活动，且无降雨干扰，利于市民和游客出行，可能带动人流量上升。  
   - **星期因素**：作为周日，属于非工作日，居民休闲需求增加，家庭出游和短途旅行可能性提高，进一步推动人流量增长。  
   - **搜索热度**：前一天景区搜索指数（904.68）高于历史平均水平（803.19），表明公众对旅游目的地的关注度提升，可能转化为实际到访人数。  

2. **总体影响判断**  
   综合上述因素，当日人流量呈现**正向增加趋势**。天气和周末效应共同作用下，叠加搜索热度的提升，预计较平日增长约**10%-15%**。但需注意，若景区承载能力有限，可能出现局部拥堵，但整体仍以小幅增长为主。
--------------------------------------------------


100%|████████████████████████████████████████████████████████████████████████████████| 92/92 [1:51:51<00:00, 72.95s/it]


所有事件已分析并编码完成，保存为 '事件Chat理解嵌入20260531_含历史搜索均值_SH.npy'，维度为：(92, 4096)


In [2]:
import re
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from modelscope import AutoModelForCausalLM, AutoTokenizer
import torch
from sklearn.metrics.pairwise import cosine_similarity


# ========== 1. 加载模型 ==========

chat_model_path = "D:\\shenxi\\Qwen\\Qwenchat8B"

tokenizer = AutoTokenizer.from_pretrained(chat_model_path)

chat_model = AutoModelForCausalLM.from_pretrained(
    chat_model_path,
    torch_dtype="auto",
    device_map="auto"
)

chat_model.eval()

embed_model = SentenceTransformer(r"D:/shenxi/Qwen/Qwen8B_embedding")


# ========== 2. 读取景区数据 ==========

csv_path = "79景点语义信息.csv"

try:
    df = pd.read_csv(csv_path, encoding="utf-8")
except UnicodeDecodeError:
    try:
        df = pd.read_csv(csv_path, encoding="gbk")
    except UnicodeDecodeError:
        df = pd.read_csv(csv_path, encoding="gb18030")
assert df.shape[1] >= 5, "CSV必须包含至少5列：景区名称、景区级别、景区类别、核心看点和特点、适宜人群"


# ========== 3. 工具函数 ==========

def clean_model_output(text: str) -> str:
    """
    清理模型输出，去掉 thinking 内容和特殊符号。
    """
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)

    text = (
        text
        .replace("<|im_end|>", "")
        .replace("<|endoftext|>", "")
        .replace("<|end|>", "")
        .strip()
    )

    return text


def generate_scenic_description(user_prompt: str) -> str:
    """
    调用 Chat 模型生成景区语义摘要。
    不输出思考内容。
    """
    messages = [
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer([text], return_tensors="pt").to(chat_model.device)

    with torch.no_grad():
        output_ids = chat_model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            repetition_penalty=1.05
        )[0]

    generated_text = tokenizer.decode(
        output_ids[len(inputs.input_ids[0]):],
        skip_special_tokens=False
    )

    content = clean_model_output(generated_text)

    return content


# ========== 4. Chat模型理解每个景区 ==========

descriptions = []

print("正在调用 Chat 模型生成每个景区的语义描述...")

for i, row in tqdm(df.iterrows(), total=len(df)):
    name = str(row.iloc[0])
    level = str(row.iloc[1])
    category = str(row.iloc[2])
    features = str(row.iloc[3])
    visitors = str(row.iloc[4])

    user_prompt = f"""
你是一名旅游分析助手。以下是一处景区的信息：

- 景区名称：{name}
- 景区级别：{level}
- 景区类别：{category}
- 核心看点和特点：{features}
- 适宜人群：{visitors}

请你用2-3句话对该景区进行简要总结，突出其主要特色和吸引点，
以便后续能够与其他景区进行语义上的比较。

要求：
1. 只输出最终摘要。
2. 不要输出推理过程。
3. 不要使用“首先、其次、综上”等过程性表达。
4. 不要输出 <think> 内容。
"""

    content = generate_scenic_description(user_prompt)
    descriptions.append(content)

    print(f"🏞️ 景区：{name}")
    print(f"📝 语义摘要：{content}")
    print("-" * 60)


# ========== 5. Embedding编码 ==========

print("\n正在进行语义编码...")

embeddings = embed_model.encode(
    descriptions,
    convert_to_numpy=True
)


# ========== 6. 计算相似度矩阵 ==========

print("\n正在计算景区之间的语义相似度矩阵...")

similarity_matrix = cosine_similarity(embeddings)


# ========== 7. 只保存相似度矩阵 ==========

scenic_names = df.iloc[:, 0].values

output_matrix = pd.DataFrame(
    similarity_matrix,
    index=scenic_names,
    columns=scenic_names
)

output_matrix.to_csv(
    "景区语义相似度矩阵.csv",
    encoding="utf-8-sig"
)

print(f"\n✅ 相似度矩阵已保存为：景区语义相似度矩阵.csv，维度：{similarity_matrix.shape}")

D:\ProgramData\anaconda3\envs\qwen8b\lib\site-packages\accelerate\utils\modeling.py:1582: UserWarning: Current model requires 128 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

正在调用 Chat 模型生成每个景区的语义描述...


  0%|                                                                                           | 0/79 [00:00<?, ?it/s]D:\ProgramData\anaconda3\envs\qwen8b\lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
D:\ProgramData\anaconda3\envs\qwen8b\lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
D:\ProgramData\anaconda3\envs\qwen8b\lib\site-packages\transformers\generation\configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You

...

 99%|████████████████████████████████████████████████████████████████████████████████▉ | 78/79 [17:28<00:13, 13.67s/it]

🏞️ 景区：龙潭公园
📝 语义摘要：龙潭公园作为4A级龙文化主题景区，以湖心岛、春节庙会和冰雪乐园为核心亮点，集市民休闲、晨练与节庆体验于一体，具有鲜明的文化特色和季节吸引力。
------------------------------------------------------------


100%|██████████████████████████████████████████████████████████████████████████████████| 79/79 [17:43<00:00, 13.46s/it]

🏞️ 景区：龙脉温泉度假村
📝 语义摘要：龙脉温泉度假村是集热带雨林馆、私汤泡池与娱乐宫于一体的4A级室内温泉宫，适合家庭休闲、周末放松及团队建设，兼具自然生态与娱乐体验的综合型温泉度假胜地。
------------------------------------------------------------

正在进行语义编码...



正在计算景区之间的语义相似度矩阵...

✅ 相似度矩阵已保存为：景区语义相似度矩阵.csv，维度：(79, 79)


##### 